# PGVector RAG Query Notebook — Admin vs User RLS Demo

This notebook demonstrates role-scoped document access using PGVector + LangChain.

- **Admin** (`redbank_admin`) can query documents from **all** collections
- **User** (`redbank_user`) can only query documents from the **`user`** collection

RLS policies on the `embeddings` table enforce this at the database level.

In [ ]:
!pip install -r requirements.txt

In [1]:
import os

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGEngine, PGVectorStore
import psycopg

# Connection parameters (match postgresql-credentials secret)
PG_HOST = os.getenv("PG_HOST", "postgresql.redbank-demo.svc.cluster.local")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DATABASE = os.getenv("PG_DATABASE", "db")

# Role credentials
ADMIN_USER = "redbank_admin"
ADMIN_PASS = "admin_pass"
USER_USER = "redbank_user"
USER_PASS = "user_pass"

# Embedding model — same as the ingestion pipeline
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5")

print("Setup complete")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 112/112 [00:00<00:00, 4363.91it/s]


Setup complete


## Admin Query — sees all collections

Connect as `redbank_admin` and perform a similarity search. Admin sees results from both `admin` and `user` collections.

In [2]:
admin_conn_str = (
    f"postgresql+psycopg://{ADMIN_USER}:{ADMIN_PASS}"
    f"@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
)
admin_engine = PGEngine.from_connection_string(url=admin_conn_str)

# Query across all documents — RLS allows admin to see every collection
admin_store = PGVectorStore.create_sync(
    engine=admin_engine,
    table_name="embeddings",
    embedding_service=embeddings,
    metadata_columns=["collection"],
)

query = "How do I reset my password?"
admin_results = admin_store.similarity_search(query, k=5)

print(f"Admin results ({len(admin_results)} docs):\n")
for i, doc in enumerate(admin_results):
    collection = doc.metadata.get("collection", "unknown")
    print(f"--- [{i+1}] collection={collection} ---")
    print(doc.page_content)
    print()

Admin results (5 docs):

--- [1] collection=user ---
Managing Your Password & Security Settings
RedBank Financial Services - Internal Document
1. Changing Your Password
To change your online banking password: log in to banking.redbank.com or open the RedBank mobile app.
Go to Settings > Security > Change Password. Enter your current password, then enter your new password
twice to confirm. Password requirements: at least 12 characters, must include at least one uppercase letter,
one lowercase letter, one number, and one special character (!@#$%^&*). Cannot reuse any of your last 10
passwords. Your password expires every 90 days and you will be prompted to change it. We recommend
using a password manager to generate and store strong, unique passwords.
2. Setting Up Two-Factor Authentication (2FA)
Two-factor authentication adds an extra layer of security to your account. To enable: go to Settings > Security
> Two-Factor Authentication. Choose your preferred method: Authenticator App (reco

## User Query — sees only `user` collection

Connect as `redbank_user` and run the same search. RLS restricts results to the `user` collection only.

In [3]:
user_conn_str = (
    f"postgresql+psycopg://{USER_USER}:{USER_PASS}"
    f"@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
)
user_engine = PGEngine.from_connection_string(url=user_conn_str)

# RLS restricts this role to only 'user' collection rows
user_store = PGVectorStore.create_sync(
    engine=user_engine,
    table_name="embeddings",
    embedding_service=embeddings,
    metadata_columns=["collection"],
)

user_results = user_store.similarity_search(query, k=5)

print(f"User results ({len(user_results)} docs):\n")
for i, doc in enumerate(user_results):
    collection = doc.metadata.get("collection", "unknown")
    print(f"--- [{i+1}] collection={collection} ---")
    print(doc.page_content)
    print()

User results (5 docs):

--- [1] collection=user ---
Managing Your Password & Security Settings
RedBank Financial Services - Internal Document
1. Changing Your Password
To change your online banking password: log in to banking.redbank.com or open the RedBank mobile app.
Go to Settings > Security > Change Password. Enter your current password, then enter your new password
twice to confirm. Password requirements: at least 12 characters, must include at least one uppercase letter,
one lowercase letter, one number, and one special character (!@#$%^&*). Cannot reuse any of your last 10
passwords. Your password expires every 90 days and you will be prompted to change it. We recommend
using a password manager to generate and store strong, unique passwords.
2. Setting Up Two-Factor Authentication (2FA)
Two-factor authentication adds an extra layer of security to your account. To enable: go to Settings > Security
> Two-Factor Authentication. Choose your preferred method: Authenticator App (recom

## RLS Proof — side-by-side comparison

Compare admin vs user result counts and verify via direct SQL that the user role cannot see admin rows.

In [4]:
print("=== RLS Proof ===\n")
print(f"Admin saw {len(admin_results)} results")
admin_collections = set(doc.metadata.get("collection") for doc in admin_results)
print(f"  Collections: {sorted(admin_collections)}")

print(f"\nUser saw {len(user_results)} results")
user_collections = set(doc.metadata.get("collection") for doc in user_results)
print(f"  Collections: {sorted(user_collections)}")

# Direct SQL proof: user cannot see admin rows
print("\n--- Direct SQL verification ---")

with psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DATABASE} "
    f"user={USER_USER} password={USER_PASS}"
) as conn:
    row = conn.execute(
        "SELECT count(*) FROM embeddings WHERE collection = 'admin'"
    ).fetchone()
    print(f"User SELECT WHERE collection='admin': {row[0]} rows (expected 0)")

with psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DATABASE} "
    f"user={ADMIN_USER} password={ADMIN_PASS}"
) as conn:
    row = conn.execute(
        "SELECT count(*) FROM embeddings WHERE collection = 'admin'"
    ).fetchone()
    print(f"Admin SELECT WHERE collection='admin': {row[0]} rows")

=== RLS Proof ===

Admin saw 5 results
  Collections: ['admin', 'user']

User saw 5 results
  Collections: ['user']

--- Direct SQL verification ---
User SELECT WHERE collection='admin': 0 rows (expected 0)
Admin SELECT WHERE collection='admin': 15 rows


## Document Count per Collection

Show how many document chunks are stored in each collection.

In [5]:
# Connect as admin to see all collections
with psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DATABASE} "
    f"user={ADMIN_USER} password={ADMIN_PASS}"
) as conn:
    rows = conn.execute(
        "SELECT collection, count(*) FROM embeddings GROUP BY collection ORDER BY collection"
    ).fetchall()

print("Document chunks per collection:")
for collection, count in rows:
    print(f"  {collection}: {count}")

Document chunks per collection:
  admin: 15
  user: 12
